In [6]:
import pandas as pd

#reading in the csv files 
IUCN_animals = pd.read_csv("IUCN_animals.csv")
GBIF_east_asia = pd.read_csv("GBIF_east_asia.csv", sep="\t")

In [4]:
IUCN_animals.head()


,assessmentId,internalTaxonId,scientificName,kingdomName,phylumName,orderName,className,familyName,genusName,speciesName,infraType,infraName,infraAuthority,authority,redlistCategory,redlistCriteria,criteriaVersion,populationTrend,scopes
0,495630,10030,Hexanchus griseus,ANIMALIA,CHORDATA,HEXANCHIFORMES,CHONDRICHTHYES,HEXANCHIDAE,Hexanchus,griseus,NaN,NaN,NaN,"(Bonnaterre, 1788)",Near Threatened,A2bd,3.1,Decreasing,Global
1,495907,10041,Heosemys annandalii,ANIMALIA,CHORDATA,TESTUDINES,REPTILIA,GEOEMYDIDAE,Heosemys,annandalii,NaN,NaN,NaN,"(Boulenger in Annandale &amp; Robinson, 1903)",Critically Endangered,A2cd+4cd,3.1,Decreasing,Global
2,497499,132523146,Hubbsina turneri,ANIMALIA,CHORDATA,CYPRINODONTIFORMES,ACTINOPTERYGII,GOODEIDAE,Hubbsina,turneri,NaN,NaN,NaN,"(de Buen, 1940)",Critically Endangered,"B1ab(i,ii,iii,iv)+2ab(i,ii,iii,iv)",3.1,Decreasing,Global
3,497550,10267,Hungerfordia pelewensis,ANIMALIA,MOLLUSCA,ARCHITAENIOGLOSSA,GASTROPODA,DIPLOMMATINIDAE,Hungerfordia,pelewensis,NaN,NaN,NaN,"Beddome, 1889",Endangered,"B1ab(ii,iii)+2ab(ii,iii)",3.1,Unknown,Global
4,498370,10767,Ictalurus australis,ANIMALIA,CHORDATA,SILURIFORMES,ACTINOPTERYGII,ICTALURIDAE,Ictalurus,australis,NaN,NaN,NaN,"(Meek, 1904)",Data Deficient,NaN,3.1,Decreasing,Global


In [7]:
GBIF_east_asia.head()

,taxonKey,scientificName,acceptedTaxonKey,acceptedScientificName,numberOfOccurrences,taxonRank,taxonomicStatus,kingdom,kingdomKey,phylum,...,classKey,order,orderKey,family,familyKey,genus,genusKey,species,speciesKey,iucnRedListCategory
0,341,Turbellaria,341.0,Turbellaria,46,CLASS,ACCEPTED,Animalia,1,Platyhelminthes,...,341.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,830,Nitrospirales,830.0,Nitrospirales,309,ORDER,ACCEPTED,Bacteria,3,Nitrospirota,...,10889728.0,Nitrospirales,830.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,982,Neogastropoda,982.0,Neogastropoda,3,ORDER,ACCEPTED,Animalia,1,Mollusca,...,225.0,Neogastropoda,982.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1445,Psittaciformes,1445.0,Psittaciformes,1,ORDER,ACCEPTED,Animalia,1,Chordata,...,212.0,Psittaciformes,1445.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2390,Scrophulariaceae,2390.0,Scrophulariaceae,471,FAMILY,ACCEPTED,Plantae,6,Tracheophyta,...,220.0,Lamiales,408.0,Scrophulariaceae,2390.0,NaN,NaN,NaN,NaN,NE


In [8]:
#Next we want to work with cleaned dataframes and only use relevant columnns to our research questions 


IUCN_cols = ['genusName','speciesName','scientificName','redlistCategory']

GBIF_cols = ['genus', 'species','iucnRedListCategory', 'numberOfOccurrences']

IUCN_filtered = IUCN_animals[IUCN_cols]

GBIF_filtered = GBIF_east_asia[GBIF_cols]

#the scientific name columns do not match between the datasets so let's correct that 

old_to_new_IUCN = {'genusName':'Genus',
                  'speciesName':'Species',
                  'scientificName':'Scientific Name',
                  'redlistCategory':'IUCN Red List Category'}

old_to_new_GBIF = {'genus':'Genus',
                  'species':'Scientific Name',
                  'iucnRedListCategory':'IUCN Red List Abbreviation', 
                  'numberOfOccurrences': 'Number of Occurrences'}

GBIF_renamed = GBIF_filtered.rename(columns = old_to_new_GBIF)

IUCN_renamed = IUCN_filtered.rename(columns = old_to_new_IUCN)

In [9]:
#converting data types 

GBIF_converted = GBIF_renamed.convert_dtypes()
IUCN_converted = IUCN_renamed.convert_dtypes()



In [10]:
#replacing the NaN values 
GBIF_converted = GBIF_converted.replace('NaN', pd.NA)
IUCN_converted = IUCN_converted.replace('NaN',pd.NA)

In [11]:
#merging the GBIF and IUCN datasets 

GBIF_IUCN_combined = pd.merge(
    left = GBIF_converted, 
    right = IUCN_converted, 
    left_on = "Scientific Name", 
    right_on = "Scientific Name")

GBIF_IUCN_combined

,Genus_x,Scientific Name,IUCN Red List Abbreviation,Number of Occurrences,Genus_y,Species,IUCN Red List Category
0,Heliocypha,Heliocypha biforata,LC,2,Heliocypha,biforata,Least Concern
1,Lathrecista,Lathrecista asiatica,LC,2,Lathrecista,asiatica,Least Concern
2,Troides,Troides magellanus,LC,7,Troides,magellanus,Least Concern
3,Byasa,Byasa impediens,LC,27,Byasa,impediens,Least Concern
4,Euprymna,Euprymna berryi,DD,2,Euprymna,berryi,Data Deficient
...,...,...,...,...,...,...,...
11132,Rallus,Rallus aquaticus,<NA>,22,Rallus,aquaticus,Least Concern
11133,Geloina,Geloina luchuana,NE,1,Geloina,luchuana,Data Deficient
11134,Ardea,Ardea alba,<NA>,8,Ardea,alba,Least Concern
11135,Coilia,Coilia mystus,<NA>,3,Coilia,mystus,Endangered


In [12]:
#Now that we have combined the datasets, we can calculate the counts of unique species
#first only include the statuses of interest to us

not_DD = GBIF_IUCN_combined['IUCN Red List Abbreviation'] != 'DD'

not_NE = GBIF_IUCN_combined['IUCN Red List Abbreviation'] != 'NE'

not_EW = GBIF_IUCN_combined['IUCN Red List Abbreviation'] != 'EW'

not_EX = GBIF_IUCN_combined['IUCN Red List Abbreviation'] != 'EX'

all_conditions = not_DD & not_NE & not_EW & not_EX

combined_filtered = GBIF_IUCN_combined[all_conditions]

unique_species_count = combined_filtered.groupby('IUCN Red List Abbreviation')['Scientific Name'].nunique()

print(unique_species_count)

proportions = unique_species_count / unique_species_count.sum()
print(proportions)

IUCN Red List Abbreviation
CR      90
EN     224
LC    5194
NT     322
VU     380
Name: Scientific Name, dtype: int64
IUCN Red List Abbreviation
CR    0.014493
EN    0.036071
LC    0.836393
NT    0.051852
VU    0.061192
Name: Scientific Name, dtype: float64


In [ ]:
#Now that we have the proportions belonging to each 